In [1]:
import os
os.chdir(r'C:\Users\Klara\retail-intelligence')

import pandas as pd
import plotly.express as px

results = pd.read_csv('data/processed/baseline_comparison.csv')

print(f"Total products evaluated: {len(results)}")
print(f"Prophet beats naive baseline: {results['BeatNaive'].sum()}/{len(results)}")
print(f"Prophet beats seasonal naive: {results['BeatSeasonalNaive'].sum()}/{len(results)}")

reliable = results[results['DataQualityFlag'] == 'OK']
print(f"\nOK products only ({len(reliable)}):")
print(f"Prophet beats naive baseline: {reliable['BeatNaive'].sum()}/{len(reliable)}")
print(f"Prophet beats seasonal naive: {reliable['BeatSeasonalNaive'].sum()}/{len(reliable)}")

Total products evaluated: 20
Prophet beats naive baseline: 7/20
Prophet beats seasonal naive: 13/20

OK products only (18):
Prophet beats naive baseline: 7/18
Prophet beats seasonal naive: 13/18


In [2]:
print("Average MAE — all 20 products:")
print(f"  Prophet MAE:          {results['Prophet_MAE'].mean():.1f}")
print(f"  Naive MAE:            {results['Naive_MAE'].mean():.1f}")
print(f"  Seasonal Naive MAE:   {results['SeasonalNaive_MAE'].mean():.1f}")

print("\nAverage MAE — 18 OK products only:")
print(f"  Prophet MAE:          {reliable['Prophet_MAE'].mean():.1f}")
print(f"  Naive MAE:            {reliable['Naive_MAE'].mean():.1f}")
print(f"  Seasonal Naive MAE:   {reliable['SeasonalNaive_MAE'].mean():.1f}")

metrics_plot = pd.DataFrame({
    'Model': ['Prophet', 'Naive Baseline', 'Seasonal Naive'] * 2,
    'Mean MAE': [
        results['Prophet_MAE'].mean(), results['Naive_MAE'].mean(), results['SeasonalNaive_MAE'].mean(),
        reliable['Prophet_MAE'].mean(), reliable['Naive_MAE'].mean(), reliable['SeasonalNaive_MAE'].mean()
    ],
    'Product Set': ['All 20 products'] * 3 + ['OK products only (18)'] * 3
})

fig = px.bar(
    metrics_plot,
    x='Model',
    y='Mean MAE',
    color='Product Set',
    barmode='group',
    title='Average MAE: Prophet vs Baselines — All Products vs OK-Only'
)
fig.show()

Average MAE — all 20 products:
  Prophet MAE:          1002.0
  Naive MAE:            851.0
  Seasonal Naive MAE:   1003.0

Average MAE — 18 OK products only:
  Prophet MAE:          336.1
  Naive MAE:            379.5
  Seasonal Naive MAE:   546.4


In [3]:
reliable_plot = reliable.copy()
reliable_plot['MAE_Diff'] = reliable_plot['Naive_MAE'] - reliable_plot['Prophet_MAE']
reliable_plot = reliable_plot.sort_values('MAE_Diff', ascending=False)

fig2 = px.bar(
    reliable_plot,
    x='StockCode',
    y='MAE_Diff',
    color='BeatNaive',
    title='Prophet Advantage Over Naive by Product (Naive MAE − Prophet MAE)',
    labels={'MAE_Diff': 'MAE improvement (positive = Prophet better)'},
    color_discrete_map={True: '#1f77b4', False: '#d62728'}
)
fig2.add_hline(y=0, line_dash='dash', line_color='gray')
fig2.show()

In [4]:
demand = pd.read_csv('data/processed/weekly_demand.csv')
demand['Week'] = pd.to_datetime(demand['Week'])

product_21977 = demand[demand['StockCode'] == '21977'].sort_values('Week')

print(product_21977[['Week', 'TotalQuantity']].tail(10))
print(f"\nLast 8 weeks (the test set):")
print(product_21977['TotalQuantity'].tail(8).describe())

          Week  TotalQuantity
638 2011-10-03            827
639 2011-10-10            146
640 2011-10-17            111
641 2011-10-24             95
642 2011-10-31             58
643 2011-11-07            145
644 2011-11-14            122
645 2011-11-21            357
646 2011-11-28            230
647 2011-12-05            170

Last 8 weeks (the test set):
count      8.000000
mean     161.000000
std       94.491118
min       58.000000
25%      107.000000
50%      133.500000
75%      185.000000
max      357.000000
Name: TotalQuantity, dtype: float64


In [5]:
forecasts = pd.read_csv('data/processed/all_forecasts.csv')
forecasts['ds'] = pd.to_datetime(forecasts['ds'])

prophet_21977 = forecasts[
    (forecasts['StockCode'] == '21977') & 
    (forecasts['ds'] >= '2011-10-10') & 
    (forecasts['ds'] <= '2011-12-05')
]
print(prophet_21977[['ds', 'yhat']])

            ds        yhat
727 2011-10-10  384.343776
728 2011-10-17  385.171299
729 2011-10-24  385.998822
730 2011-10-31  386.826345
731 2011-11-07  387.653868
732 2011-11-14  388.481391
733 2011-11-21  389.308914
734 2011-11-28  390.136437
735 2011-12-05  390.963960


# Day 4 — Notes & Observations

**Data loading cell** — loaded `baseline_comparison.csv` and compared results using all 20 products versus only the 18 "OK" products. The conclusions are unchanged: Prophet beats naive on 7 of 18 reliable products and beats seasonal naive on 13 of 18 reliable products. The two flagged products do not change the win/loss pattern, but they strongly inflate average error metrics.

**Average MAE comparison** — confirmed the earlier script's finding: including the two flagged products inflates Prophet's average MAE from 336.1 (OK products only) to 1002.0 (all 20 products), nearly a threefold increase. The extreme forecasts of products 23843 and 23166 distort any simple average, making a strong case for reporting metrics both with and without flagged products rather than relying on a single blended number.

**Per-product MAE improvement chart** — visualizes the win/loss pattern directly. Prophet wins by a large margin on a handful of products (22197: +540 units, 84879, 84077, 22616), while losing by relatively small margins on most others, except for one clear outlier: 21977, investigated below.

---

## Key finding: Prophet achieves larger wins, but on fewer products

At first glance, Prophet's lower average MAE (336.1 versus 379.5 for naive) suggests that it is the stronger model overall. However, the win rate tells a different story: Prophet only beats naive on 7 of the 18 reliable products.

This means Prophet's advantage comes from a small number of very large improvements rather than consistently outperforming naive across the dataset. Naive remains slightly better for most products, even though Prophet achieves the lower average error.

This is an important finding rather than a failure. Average performance and per-product consistency measure different aspects of model quality, so both metrics belong in the final evaluation rather than only whichever one produces the more favorable result.

---

## Why Prophet beats seasonal naive more easily than plain naive

Seasonal naive predicts "the same week as last year" using only a single historical observation. Because this dataset contains only about one year of history, seasonal naive cannot reliably estimate recurring yearly patterns.

This limitation likely explains why Prophet outperforms seasonal naive much more often (13 of 18 products) than plain naive (7 of 18 products). The result should therefore be interpreted as Prophet outperforming a weak attempt at modeling seasonality, rather than demonstrating an unconditional advantage over simple baselines.

---

## Sanity check: why does Prophet lose worst on 21977?

To investigate, actual values and Prophet forecasts were extracted directly from the test window:

| Week | Actual | Prophet yhat |
|---|---:|---:|
| 2011-10-17 | 111 | 385.2 |
| 2011-10-24 | 95 | 386.0 |
| 2011-10-31 | 58 | 386.8 |
| 2011-11-07 | 145 | 387.7 |
| 2011-11-14 | 122 | 388.5 |
| 2011-11-21 | 357 | 389.3 |
| 2011-11-28 | 230 | 390.1 |
| 2011-12-05 | 170 | 391.0 |

Prophet forecasted a nearly flat 385–391 units per week across the entire test period, while actual demand ranged from 58 to 357 units (mean = 161). By contrast, naive's flat forecast of 146 units—the final value from the training period—happened to lie much closer to the true demand range.

The likely cause is that product 21977 was classified as having a growing trend (see Day 2 analysis). With no weekly or yearly seasonality components available, the trend line continued upward even though demand flattened during the test period.

Importantly, this is a different problem from the sparse-data issue captured by `DataQualityFlag`. A product can satisfy the data-quality criteria and still produce a poor forecast because of trend overshoot.

This highlights an important limitation of the flagging system: `DataQualityFlag` detects sparse and spike-driven demand patterns, but it does not capture all possible forecasting failures.

**Not fixed today.** This notebook evaluates only a single 8-week holdout period, and tuning `changepoint_prior_scale` or switching to logistic growth based on one split would risk overfitting to this specific window.

The question is therefore deferred to Week 5's walk-forward validation, which will determine whether this overshoot represents a genuine, repeated pattern across multiple time windows or simply an artifact of this particular test split.

Potential future fixes, if supported by the Week 5 results:

- Reduce `changepoint_prior_scale`
- Test logistic growth with a demand cap

---

## Caveat: this is a simplified comparison, not yet rigorous

Per the original Week 4 plan, today's evaluation uses a single train/test split that holds out the final eight weeks without retraining.

Full walk-forward validation, which repeatedly retrains and evaluates the model across multiple rolling windows, is introduced in Week 5.

As a result, today's conclusions—including the apparent failure on product 21977—should be treated as preliminary findings rather than definitive evidence. Some patterns observed here may disappear, strengthen, or reverse once tested across multiple time periods instead of a single holdout window.